
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.7_flash_attention_practice/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.7_flash_attention_practice/lab.ipynb)

# 3.7 Lab: FlashAttention in Practice

**Goal**: Use FlashAttention through PyTorch's standard APIs and measure its impact at both kernel and model level.



In [ ]:
# --- Setup: install dependencies ---
!pip install -q torch transformers accelerate matplotlib

import torch
import time
import matplotlib.pyplot as plt
from torch.nn.functional import scaled_dot_product_attention

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.get_device_name(0)}")
print(f"Compute capability: {torch.cuda.get_device_capability()}")


## 1. PyTorch SDPA Backends

`torch.nn.functional.scaled_dot_product_attention` dispatches to one of three kernels:
- **flash** — FlashAttention-2 (fused, IO-aware, no materialized attention matrix)
- **mem_efficient** — xFormers memory-efficient attention
- **math** — naive PyTorch implementation (materializes full NxN matrix)

You select backends using context managers.


In [ ]:
# --- Demonstrate backend selection with context managers ---
from torch.backends.cuda import (
    sdp_kernel,  # context manager to enable/disable backends
    flash_sdp_enabled,
    mem_efficient_sdp_enabled,
    math_sdp_enabled,
)

# Create sample tensors: (batch, heads, seq_len, head_dim)
B, H, S, D = 2, 32, 1024, 128
q = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
k = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
v = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)

# Force flash backend only
with sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    out_flash = scaled_dot_product_attention(q, k, v)

# Force math backend only
with sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=False):
    out_math = scaled_dot_product_attention(q, k, v)

# Verify outputs match (within fp16 tolerance)
diff = (out_flash - out_math).abs().max().item()
print(f"Max difference flash vs math: {diff:.6f}")
print(f"Shape: {out_flash.shape}")


In [ ]:
# --- Benchmark utility: measure kernel latency and peak memory ---
def benchmark_sdp_kernel(q, k, v, backend, n_warmup=10, n_iter=50):
    """Time a single SDPA backend over n_iter forward passes."""
    enable = {"enable_flash": False, "enable_math": False, "enable_mem_efficient": False}
    enable[f"enable_{backend}"] = True

    # Warmup
    with sdp_kernel(**enable):
        for _ in range(n_warmup):
            _ = scaled_dot_product_attention(q, k, v)
    torch.cuda.synchronize()

    # Measure
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with sdp_kernel(**enable):
        for _ in range(n_iter):
            _ = scaled_dot_product_attention(q, k, v)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_iter * 1000  # ms per call

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return elapsed, peak_mb


In [ ]:
# --- Benchmark all backends across sequence lengths ---
SEQ_LENGTHS = [512, 1024, 2048, 4096, 8192]
BACKENDS = ["flash", "mem_efficient", "math"]
B, H, D = 2, 32, 128  # batch=2, heads=32, head_dim=128

results = {b: {"latency": [], "memory": []} for b in BACKENDS}

for seq_len in SEQ_LENGTHS:
    q = torch.randn(B, H, seq_len, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, seq_len, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, seq_len, D, device="cuda", dtype=torch.float16)

    for backend in BACKENDS:
        try:
            lat, mem = benchmark_sdp_kernel(q, k, v, backend)
            results[backend]["latency"].append(lat)
            results[backend]["memory"].append(mem)
        except RuntimeError as e:
            # Backend not supported on this GPU
            print(f"  {backend} @ seq={seq_len}: SKIPPED ({e})")
            results[backend]["latency"].append(None)
            results[backend]["memory"].append(None)

    # Free memory between iterations
    del q, k, v
    torch.cuda.empty_cache()

print("Kernel benchmarks complete.")


In [ ]:
# --- Plot: kernel latency and memory vs sequence length ---
fig_4, (ax1_4, ax2_4) = plt.subplots(1, 2, figsize=(12, 5))
colors = {"flash": "#2563eb", "mem_efficient": "#16a34a", "math": "#dc2626"}

for backend in BACKENDS:
    lats = results[backend]["latency"]
    mems = results[backend]["memory"]
    # Filter None values
    valid = [(s, l, m) for s, l, m in zip(SEQ_LENGTHS, lats, mems) if l is not None]
    if not valid:
        continue
    seqs, lats_v, mems_v = zip(*valid)

    ax1_4.plot(seqs, lats_v, "o-", label=backend, color=colors[backend], linewidth=2)
    ax2_4.plot(seqs, mems_v, "o-", label=backend, color=colors[backend], linewidth=2)

ax1_4.set_xlabel("Sequence Length")
ax1_4.set_ylabel("Latency (ms)")
ax1_4.set_title("SDPA Kernel Latency")
ax1_4.legend()
ax1_4.set_xscale("log", base=2)
ax1_4.grid(True, alpha=0.3)

ax2_4.set_xlabel("Sequence Length")
ax2_4.set_ylabel("Peak Memory (MB)")
ax2_4.set_title("SDPA Kernel Memory")
ax2_4.legend()
ax2_4.set_xscale("log", base=2)
ax2_4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kernel_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Key insight: math backend scales O(N^2) in memory; flash stays O(N).")


## 2. Model-Level Benchmark: Mistral-7B

Compare `attn_implementation="eager"` (math kernel) vs `attn_implementation="flash_attention_2"` on a real model.


In [ ]:
# --- Load Mistral-7B with both attention implementations ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

MODEL_ID = "mistralai/Mistral-7B-v0.1"

# Load tokenizer (shared)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Load with eager attention (baseline)
print("Loading eager model...")
model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="eager"
)

# Load with flash_attention_2
print("Loading flash model...")
model_flash = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="flash_attention_2"
)
print("Both models loaded.")


In [ ]:
# --- Benchmark model forward pass: latency + peak memory ---
def benchmark_model_forward(model, input_ids, n_warmup=3, n_iter=10):
    """Measure forward pass latency and peak GPU memory."""
    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(input_ids)
    torch.cuda.synchronize()

    # Measure
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iter):
            _ = model(input_ids)
    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) / n_iter * 1000

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return latency_ms, peak_mb

# Run at multiple sequence lengths
MODEL_SEQ_LENGTHS = [128, 512, 1024, 2048]
model_results = {"eager": {"latency": [], "memory": []}, "flash": {"latency": [], "memory": []}}

for seq_len in MODEL_SEQ_LENGTHS:
    # Generate dummy input
    input_ids = torch.randint(0, 32000, (1, seq_len), device="cuda")

    # Eager
    torch.cuda.empty_cache()
    lat_e, mem_e = benchmark_model_forward(model_eager, input_ids)
    model_results["eager"]["latency"].append(lat_e)
    model_results["eager"]["memory"].append(mem_e)

    # Flash
    torch.cuda.empty_cache()
    lat_f, mem_f = benchmark_model_forward(model_flash, input_ids)
    model_results["flash"]["latency"].append(lat_f)
    model_results["flash"]["memory"].append(mem_f)

    speedup = lat_e / lat_f
    mem_saved = (mem_e - mem_f) / mem_e * 100
    print(f"seq={seq_len:>5} | eager: {lat_e:.1f}ms, {mem_e:.0f}MB | "
          f"flash: {lat_f:.1f}ms, {mem_f:.0f}MB | "
          f"speedup: {speedup:.2f}x, mem saved: {mem_saved:.1f}%")


In [ ]:
# --- Plot: model-level latency and memory comparison ---
fig_7, (ax1_7, ax2_7) = plt.subplots(1, 2, figsize=(12, 5))

ax1_7.plot(MODEL_SEQ_LENGTHS, model_results["eager"]["latency"], "o-",
         label="eager", color="#dc2626", linewidth=2)
ax1_7.plot(MODEL_SEQ_LENGTHS, model_results["flash"]["latency"], "o-",
         label="flash_attention_2", color="#2563eb", linewidth=2)
ax1_7.set_xlabel("Sequence Length")
ax1_7.set_ylabel("Forward Pass Latency (ms)")
ax1_7.set_title("Mistral-7B: Eager vs Flash Attention")
ax1_7.legend()
ax1_7.grid(True, alpha=0.3)

ax2_7.plot(MODEL_SEQ_LENGTHS, model_results["eager"]["memory"], "o-",
         label="eager", color="#dc2626", linewidth=2)
ax2_7.plot(MODEL_SEQ_LENGTHS, model_results["flash"]["memory"], "o-",
         label="flash_attention_2", color="#2563eb", linewidth=2)
ax2_7.set_xlabel("Sequence Length")
ax2_7.set_ylabel("Peak GPU Memory (MB)")
ax2_7.set_title("Mistral-7B: Memory Usage")
ax2_7.legend()
ax2_7.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("model_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()


## Key Takeaways

1. **Using FlashAttention requires zero code changes** — just set `attn_implementation="flash_attention_2"` in HuggingFace or use `sdp_kernel(enable_flash=True)` with raw PyTorch SDPA.

2. **Kernel-level**: Flash and mem_efficient scale O(N) in memory vs O(N²) for math. Latency gap widens dramatically beyond seq_len=2048.

3. **Model-level**: On Mistral-7B, flash_attention_2 delivers 1.5-2.5x speedup and 20-40% memory savings at seq_len≥1024. The savings fund either longer contexts or larger batch sizes.

4. **When to use which**:
   - `flash` — production serving, long contexts (requires SM>=80)
   - `mem_efficient` — fallback for older GPUs (T4, V100)
   - `math` — debugging only (materializes full attention matrix)
